# Modeling and evaluation
Run from the repository root or notebooks directory. The training entry point regenerates all artifacts and evaluates its final winner once.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
import pandas as pd
import joblib
from src.features import DATA, MODELS, FEATURES
from src.train import main
from src.explain import explain_single_prediction

## Compare and tune
Each fold fits its own preprocessing. The baseline averages INR directly. Four regressors learn log prices. The lowest validation MAE determines which candidate enters a 15-draw search on the same folds. The better of the default and tuned configuration advances.

In [ ]:
results = main()
pd.DataFrame(results['comparison']).set_index('model').round(3)

## Final holdout
The figures below reuse the single holdout prediction run. MAE and RMSE measure INR error. R² is dimensionless.

In [ ]:
display(results['winner'], results['parameters'], results['test'])

## Price drivers
Global importance measures the increase in training MAE after permutation. Local effects sum encoded columns into their original feature groups and describe changes in log price.

In [ ]:
model = joblib.load(MODELS / "wheelvalue_pipeline.joblib")
display(pd.read_json(MODELS / "feature_importance.json", typ="series"))
example = pd.read_csv(DATA / "train.csv", nrows=1)[FEATURES]
explain_single_prediction(model, example)